In [1]:
# Install Huggingface Transformers if needed
!pip install transformers 

# Install torch if not installed
!pip install torch 

  Using cached transformers-4.51.3-py3-none-any.whl.metadata (38 kB)
  Using cached filelock-3.18.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached huggingface_hub-0.30.2-py3-none-any.whl.metadata (13 kB)
  Using cached tokenizers-0.21.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.8 kB)
  Using cached safetensors-0.5.3-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.8 kB)
  Using cached fsspec-2025.3.2-py3-none-any.whl.metadata (11 kB)
Using cached transformers-4.51.3-py3-none-any.whl (10.4 MB)
Using cached huggingface_hub-0.30.2-py3-none-any.whl (481 kB)
Using cached safetensors-0.5.3-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (471 kB)
Using cached tokenizers-0.21.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.0 MB)
Using cached filelock-3.18.0-py3-none-any.whl (16 kB)
Using cached fsspec-2025.3.2-py3-none-any.whl (194 kB)
  Using cached torch-2.7.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Usi

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

In [5]:
# Load FinBERT model and tokenizer
#model_name = "yiyanghkust/finbert-tone"
model_name = "ProsusAI/finbert"
#model_name = "ipuneetrathore/bert-base-cased-financial-news-sentiment-analysis"
#model_name = "amphora/bert-base-finance-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [6]:
def predict_financial_sentiment(text):
    # Tokenize input
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    
    # Get model outputs
    with torch.no_grad():
        outputs = model(**inputs)
    
    logits = outputs.logits
    probs = F.softmax(logits, dim=1)

    # Map prediction
    labels = ['positive', 'neutral', 'negative']
    predicted_class = torch.argmax(probs).item()
    predicted_label = labels[predicted_class]
    
    # Also return probabilities for each class
    probs_dict = {label: prob.item() for label, prob in zip(labels, probs.squeeze())}

    return predicted_label, probs_dict

In [7]:
# Example financial sentence
#text = "The company's quarterly profits exceeded expectations, boosting investor confidence."
text = ("The company posted record-breaking profits this quarter, delighting shareholders.")
#text = ("The company declared bankruptcy and investors lost everything.")
#text = ("Tariffs cause loss of export market")
#text = ("Tariffs protect domestic market for steel from cheap imports")
#text = ("Huge popularity leads to surge in demand")


# Predict sentiment
label, probs = predict_financial_sentiment(text)

print(f"Predicted Sentiment: {label}")
print("Class Probabilities:")
for k, v in probs.items():
    print(f"{k}: {v:.4f}")

Predicted Sentiment: positive
Class Probabilities:
positive: 0.9440
neutral: 0.0189
negative: 0.0371


In [8]:
predict_financial_sentiment("The company achieved exceptional revenue growth and increased shareholder value.")

('positive',
 {'positive': 0.9560390710830688,
  'neutral': 0.016260437667369843,
  'negative': 0.027700403705239296})

In [9]:
predict_financial_sentiment("The company's stock price jumped 20% after reporting massive earnings growth.")

('positive',
 {'positive': 0.9490567445755005,
  'neutral': 0.019320795312523842,
  'negative': 0.03162248805165291})

In [10]:
corpus = [
    "Apple shares jump as company reports record earnings.",
    "Tesla stock falls after CEO warns of supply chain issues.",
    "Microsoft announces major dividend increase for investors.",
    "Amazon faces antitrust investigation from European regulators.",
    "Meta sees user growth slow, but beats earnings expectations.",
    "Oil prices rise as demand surges post-pandemic.",
    "Federal Reserve signals more interest rate hikes ahead.",
    "Netflix stock plunges after losing subscribers.",
    "Google parent Alphabet's profits soar beyond Wall Street forecasts.",
    "Gold prices steady amid inflation concerns."
]

In [11]:
import pandas as pd

def predict_corpus_sentiment(corpus):
    results = []

    for text in corpus:
        inputs = tokenizer(text, return_tensors="pt", truncation=True)
        with torch.no_grad():
            outputs = model(**inputs)
        
        logits = outputs.logits
        probs = F.softmax(logits, dim=1)
        
        labels = ['positive', 'neutral', 'negative']
        predicted_class = torch.argmax(probs).item()
        predicted_label = labels[predicted_class]
        
        probs_dict = {label: prob.item() for label, prob in zip(labels, probs.squeeze())}
        
        results.append({
            'text': text,
            'predicted_sentiment': predicted_label,
            'positive_prob': probs_dict['positive'],
            'neutral_prob': probs_dict['neutral'],
            'negative_prob': probs_dict['negative']
        })
    
    return pd.DataFrame(results)

In [12]:
df_results = predict_corpus_sentiment(corpus)
df_results

,text,predicted_sentiment,positive_prob,neutral_prob,negative_prob
0,Apple shares jump as company reports record ea...,negative,0.285943,0.336935,0.377122
1,Tesla stock falls after CEO warns of supply ch...,neutral,0.013923,0.942455,0.043622
2,Microsoft announces major dividend increase fo...,positive,0.769775,0.028358,0.201867
3,Amazon faces antitrust investigation from Euro...,neutral,0.009965,0.925089,0.064946
4,"Meta sees user growth slow, but beats earnings...",neutral,0.398223,0.524415,0.077362
5,Oil prices rise as demand surges post-pandemic.,neutral,0.301238,0.621045,0.077717
6,Federal Reserve signals more interest rate hik...,positive,0.404745,0.197854,0.397401
7,Netflix stock plunges after losing subscribers.,neutral,0.007548,0.966514,0.025937
8,Google parent Alphabet's profits soar beyond W...,positive,0.903001,0.063816,0.033182
9,Gold prices steady amid inflation concerns.,positive,0.762235,0.149847,0.087917
